## Building the factor universe and the scoring pipeline

Per the README's build order, this notebook builds and validates every raw factor (the five
foundational ones first, then the rest of the JKP taxonomy and the novel alpha candidates as
they're built) and the scoring pipeline that operates on top of whatever factors exist
(`scoring/zscore.py`, `scoring/combine.py`, `scoring/neutralize.py`), before any of it gets
promoted into `src/`.

Part 3 (the factors) grows over time as new ones are built; Part 4 (scoring) stays last,
since it's generic machinery that consumes the factor set rather than something tied to a
fixed number of them.

Each part follows the same shape: build a small piece, validate it against a toy case with a
known answer, then check it against a real sample before it's trusted enough to promote. Real
bugs found along the way (a ticker misattribution in `ticker_on`, a stock-split gap in market
cap) are fixed at the source and documented in `notebooks/logs/` and the relevant module's own
docstring, not replayed here; this notebook stays focused on validating what exists today.

## Part 1: setup

Universe, fundamentals, and price caches, plus a fixed, reproducible sample of real companies
reused throughout the rest of this notebook for real-data checks.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import math
import random

import pandas as pd

from src.universe.point_in_time import build_universe, ticker_on
from src.loaders.fundamentals import build_fundamentals, load_company_facts
from src.loaders.prices import build_prices, load_cik_prices, close_on_or_before
from src.factors.size import market_cap_as_of, split_adjustment_ratio, size_factor
from src.factors.value import earnings_yield_factor
from src.factors.quality import roe_factor
from src.factors.momentum import momentum_factor
from src.factors.low_vol import low_vol_factor
from src.scoring.zscore import winsorize, zscore
from src.scoring.combine import combine
from src.scoring.neutralize import neutralize

universe_spans, ticker_history = build_universe()
build_fundamentals()
build_prices()

AS_OF = "2024-06-28"   # a recent Friday, arbitrary
random.seed(0)          # fixed, so real-data checks below are reproducible run to run

active = universe_spans[
    (universe_spans["start_date"] <= AS_OF)
    & (universe_spans["end_date"].isna() | (universe_spans["end_date"] >= AS_OF))
]
sample_ciks = random.sample(list(active["cik"].dropna().unique()), 60)

## Part 2: price and market cap helpers

`close_on_or_before` (point in time close, never a later date), `split_adjustment_ratio`, and
`market_cap_as_of` now live in `src/loaders/prices.py` and `src/factors/size.py`. The real bug
that motivated `split_adjustment_ratio`: a filed share count and a cached price come from two
different sources with two different split bases, and combining them directly silently
understated market cap by the cumulative split ratio for any company that split its stock
between the filing and today. Confirmed on CMG (a real 50-for-1 split) and ORLY, whose split
happened chronologically *after* `AS_OF` and still corrupted the result, since cached prices
are always adjusted to whenever they were fetched, not to `AS_OF`.

In [ ]:
toy_prices = pd.DataFrame({
    "ticker": ["XYZ", "XYZ", "XYZ"],
    "Close": [10.0, 11.0, 12.0],
}, index=pd.to_datetime(["2024-01-05", "2024-01-08", "2024-01-09"]).tz_localize("America/New_York"))
# 2024-01-05 is a Friday, 2024-01-08 a Monday: a real weekend gap in between.

print(close_on_or_before(toy_prices, "XYZ", "2024-01-08"))  # exact day: expect 11.0
print(close_on_or_before(toy_prices, "XYZ", "2024-01-07"))  # a Sunday: expect 10.0, Friday's close
print(close_on_or_before(toy_prices, "XYZ", "2024-01-01"))  # before any data: expect None
print(close_on_or_before(toy_prices, "ABC", "2024-01-08"))  # ticker not present: expect None

cmg_prices = load_cik_prices(1058090)
print(split_adjustment_ratio(cmg_prices, "CMG", "2024-04-22"))    # real 50-for-1 split; expect 50.0

orly_prices = load_cik_prices(898173)
print(split_adjustment_ratio(orly_prices, "ORLY", "2024-04-29"))  # expect 15.0, split after AS_OF

aapl_prices = load_cik_prices(320193)
print(split_adjustment_ratio(aapl_prices, "AAPL", "2024-06-28"))  # expect 1.0: no split since 2020

## Part 3: the factors

Each raw factor returns the plain quantity it's named for (log market cap, an earnings yield,
a return on equity, a trailing return, a standard deviation), never negated or rescaled for its
expected direction of alpha. Which direction to bet is a decision for the alpha model and
optimizer later, not something baked into a raw factor's sign here.

3a through 3e are the five foundational factors; later subparts continue with the rest of the
JKP taxonomy and the novel alpha candidates as they're built, per the README's build order.

### 3a. Size: log market capitalization

In [ ]:
print(size_factor(math.e))   # log(e) = 1, exact
print(size_factor(1.0))      # log(1) = 0, exact
print(size_factor(None))     # expect None
print(size_factor(-100.0))   # expect None, defensive

rows = []
for cik in sample_ciks:
    facts = load_company_facts(cik)
    ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    if facts is None or ticker is None or prices is None:
        continue
    sf = size_factor(market_cap_as_of(facts, prices, ticker, AS_OF))
    if sf is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "size_factor": sf})

size_df = pd.DataFrame(rows)
print(f"{len(size_df)} / {len(sample_ciks)} resolved")
print(size_df["size_factor"].describe())

### 3b. Value: earnings yield

Net income over split-adjusted market cap. Earnings, not revenue or gross profit, as the
numerator, since `net_income` is tagged consistently across nearly every filer where the other
two have documented sector gaps (banks report interest income instead of revenue).

In [ ]:
toy_value_facts = {
    "facts": {
        "dei": {
            "EntityCommonStockSharesOutstanding": {
                "units": {"shares": [{"end": "2024-04-22", "val": 1000.0, "filed": "2024-04-25", "form": "10-Q"}]}
            }
        },
        "us-gaap": {
            "NetIncomeLoss": {
                "units": {"USD": [
                    {"start": "2023-01-01", "end": "2023-12-31", "val": 500.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            }
        },
    }
}
toy_value_prices = pd.DataFrame({
    "ticker": ["XYZ", "XYZ", "XYZ"],
    "Close": [100.0, 2.0, 2.1],
    "Stock Splits": [0.0, 50.0, 0.0],
}, index=pd.to_datetime(["2024-04-22", "2024-06-26", "2024-06-28"]).tz_localize("America/New_York"))

result = earnings_yield_factor(toy_value_facts, toy_value_prices, "XYZ", "2024-06-28")
print(result)   # expect 500.0 / (1000.0 * 50.0 * 2.1) = 500.0 / 105000.0 ≈ 0.004762

rows = []
for cik in sample_ciks:
    facts = load_company_facts(cik)
    ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    if facts is None or ticker is None or prices is None:
        continue
    ey = earnings_yield_factor(facts, prices, ticker, AS_OF)
    if ey is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "earnings_yield": ey})

value_df = pd.DataFrame(rows)
print(f"{len(value_df)} / {len(sample_ciks)} resolved")
print(value_df["earnings_yield"].describe())

### 3c. Quality: return on equity

Negative equity (leveraged buybacks) is deliberately not filtered out: it's real, common data,
not an error, left for `scoring/zscore.py`'s winsorization and later IC measurement to handle.

In [ ]:
toy_quality_facts = {
    "facts": {
        "us-gaap": {
            "NetIncomeLoss": {
                "units": {"USD": [
                    {"start": "2023-01-01", "end": "2023-12-31", "val": 500.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
            "StockholdersEquity": {
                "units": {"USD": [
                    {"end": "2023-12-31", "val": 2500.0, "filed": "2024-02-01", "form": "10-K"},
                ]}
            },
        }
    }
}
print(roe_factor(toy_quality_facts, "2024-06-28"))   # expect 500.0 / 2500.0 = 0.2

rows = []
for cik in sample_ciks:
    facts = load_company_facts(cik)
    if facts is None:
        continue
    ticker = ticker_on(ticker_history, cik, AS_OF)
    roe = roe_factor(facts, AS_OF)
    if roe is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "roe": roe})

quality_df = pd.DataFrame(rows)
print(f"{len(quality_df)} / {len(sample_ciks)} resolved")
print(quality_df["roe"].describe())

### 3d. Momentum: trailing 12 month return, skipping the most recent month

The most recent month is skipped deliberately: short term reversal works in the opposite
direction to momentum over roughly a one month horizon, so including it would blend two
factors with opposite signs into one noisy signal.

In [ ]:
toy_momentum_prices = pd.DataFrame({
    "ticker": ["XYZ", "XYZ"],
    "Close": [100.0, 150.0],
}, index=pd.to_datetime(["2023-06-28", "2024-05-28"]).tz_localize("America/New_York"))
print(momentum_factor(toy_momentum_prices, "XYZ", "2024-06-28"))   # expect 150/100 - 1 = 0.5

rows = []
for cik in sample_ciks:
    ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    if ticker is None or prices is None:
        continue
    mom = momentum_factor(prices, ticker, AS_OF)
    if mom is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "momentum": mom})

momentum_df = pd.DataFrame(rows)
print(f"{len(momentum_df)} / {len(sample_ciks)} resolved")
print(momentum_df["momentum"].describe())

### 3e. Low volatility: standard deviation of trailing daily returns

252 trading days (about a year), matching momentum's own lookback: more observations give a
materially less noisy standard deviation estimate. Returns the raw standard deviation, not its
negative, same raw-quantity convention as every other factor above.

In [ ]:
toy_lowvol_prices = pd.DataFrame({
    "ticker": ["XYZ"] * 5,
    "Close": [100.0, 110.0, 99.0, 108.9, 98.01],
}, index=pd.date_range("2024-06-24", periods=5, freq="B").tz_localize("America/New_York"))
result = low_vol_factor(toy_lowvol_prices, "XYZ", "2024-06-28", lookback_days=4)
expected = pd.Series([0.1, -0.1, 0.1, -0.1]).std()
print(result, expected)   # both should match (small floating point noise aside)

rows = []
for cik in sample_ciks:
    ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    if ticker is None or prices is None:
        continue
    vol = low_vol_factor(prices, ticker, AS_OF)
    if vol is None:
        continue
    rows.append({"cik": cik, "ticker": ticker, "low_vol": vol})

lowvol_df = pd.DataFrame(rows)
print(f"{len(lowvol_df)} / {len(sample_ciks)} resolved")
print(lowvol_df["low_vol"].describe())

### Panel check: known hard cases

`notebooks/panel.py`'s 28-company panel (shared with `validating_fundamentals.ipynb`) exists
specifically because a random sample can go a long time without ever hitting a genuinely hard
case: a dual class share count (`GOOGL`, `META`), a tag naming era transition (`CSX`, `HD`,
`TSN`), or a filer that returns `None` for everything (`CCU-200807`, a foreign private issuer).
Running every factor against it is a different, complementary check from the random-sample
`describe()` calls above: not "is the distribution sane" but "does a known-tricky company
resolve to `None` where it should, and a real number where it shouldn't, without crashing."

In [ ]:
from notebooks.panel import resolve_panel_ciks

panel_rows = []
for ticker, cik, axis, why in resolve_panel_ciks(ticker_history):
    facts = load_company_facts(cik)
    p_ticker = ticker_on(ticker_history, cik, AS_OF)
    prices = load_cik_prices(cik)
    have_price_inputs = prices is not None and p_ticker is not None
    market_cap = market_cap_as_of(facts, prices, p_ticker, AS_OF) if facts is not None and have_price_inputs else None

    panel_rows.append({
        "ticker": ticker,
        "axis": axis,
        "size": size_factor(market_cap),
        "value": earnings_yield_factor(facts, prices, p_ticker, AS_OF) if facts is not None and have_price_inputs else None,
        "quality": roe_factor(facts, AS_OF) if facts is not None else None,
        "momentum": momentum_factor(prices, p_ticker, AS_OF) if have_price_inputs else None,
        "low_vol": low_vol_factor(prices, p_ticker, AS_OF) if have_price_inputs else None,
    })

panel_df = pd.DataFrame(panel_rows).set_index("ticker")
panel_df

### Summary: all factors built so far

One combined table and one chart, both meant to be extended, not rebuilt, as new factors are
added: append a new key to `factor_series` below and both the table and the chart pick it up
automatically.

In [ ]:
factor_series = {
    "size": size_df.set_index("cik")["size_factor"],
    "value": value_df.set_index("cik")["earnings_yield"],
    "quality": quality_df.set_index("cik")["roe"],
    "momentum": momentum_df.set_index("cik")["momentum"],
    "low_vol": lowvol_df.set_index("cik")["low_vol"],
}

summary = pd.DataFrame({name: s.describe() for name, s in factor_series.items()})
summary

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(factor_series), figsize=(4 * len(factor_series), 3.2))
for ax, (name, series) in zip(axes, factor_series.items()):
    ax.hist(series.dropna(), bins=20, color="#3987e5", edgecolor="white", linewidth=0.5)
    ax.set_title(name)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle(f"Factor distributions, {len(sample_ciks)}-company sample, {AS_OF}")
fig.tight_layout()
plt.show()

## Part 4: scoring — zscore, combine, neutralize

### 4a. Cross sectional z-score, with winsorization

Whether z-scoring needs outlier handling was checked empirically rather than assumed: 3b's
real earnings-yield sample shows no outlier disconnected from its neighbors once the
split-adjustment bug was fixed, so winsorizing at the 1st/99th percentile is cheap insurance
for a factor or date not covered by that specific check (3c's ROE sample has a much wider
spread), not a correction this data was shown to need on its own.

In [ ]:
toy = pd.Series(range(1, 101), dtype=float)               # 1..100, evenly spread
toy_with_outlier = pd.concat([toy, pd.Series([10_000.0])], ignore_index=True)
clipped = winsorize(toy_with_outlier, 0.01, 0.99)
print("raw max:", toy_with_outlier.max(), " clipped max:", clipped.max())

toy_with_gap = pd.Series([1.0, 2.0, 3.0, None, 5.0])
z = zscore(toy_with_gap, winsorize_pct=0)
print(z)
print("NaN count:", z.isna().sum())

### 4b. Combine: weighted average with missing-data handling

A stock missing one or more factors has those terms dropped and the remaining weights
renormalized, per the README's missing-data rule, never a substituted zero.

In [ ]:
factors = pd.DataFrame({
    "momentum": [1.0, 1.0],
    "value": [-1.0, None],
}, index=["A", "B"])
weights = {"momentum": 0.5, "value": 0.5}
print(combine(factors, weights))   # expect A: 0.0, B: 1.0, not 0.5

factors_all_missing = pd.DataFrame({
    "momentum": [1.0, None],
    "value": [-1.0, None],
}, index=["A", "B"])
print(combine(factors_all_missing, weights))   # expect A: 0.0, B: NaN, not an error

### 4c. Neutralize: residualize the combined score against beta

Refit fresh at every rebalance date, never reused, since the regression coefficients are
expected to change from one date to the next.

In [ ]:
betas = pd.Series([1.0, 2.0, 3.0, 4.0], index=["A", "B", "C", "D"])
scores = pd.Series([1.0, 2.0, 6.0, None], index=["A", "B", "C", "D"])
# A, B, C fit a line with slope 2.5, intercept -2: predicted 0.5, 3.0, 5.5,
# residuals 0.5, -1.0, 0.5. D has a beta but no score, excluded from the fit.
print(neutralize(scores, betas))   # expect A: 0.5, B: -1.0, C: 0.5, D: NaN